In [0]:

CREATE SCHEMA IF NOT EXISTS weather_openmeteo.silver;

use weather_openmeteo.silver;

CREATE TABLE IF NOT EXISTS weather_openmeteo.silver.weather_clean (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    temperature DOUBLE
)
USING DELTA;

-- 1. Deduplicate and clean the raw Bronze data using a CTE
WITH source_cleaned AS (
    SELECT 
        CAST(date AS TIMESTAMP) AS date,
        CAST(latitude AS DOUBLE) AS latitude,
        CAST(longitude AS DOUBLE) AS longitude,
        CAST(temperature_2m AS DOUBLE) AS temperature,
        
        -- Conditional mapping logic to assign city names based on coordinates
        CASE 
            WHEN ROUND(latitude, 4) = 42.4955 AND ROUND(longitude, 4) = -89.0273 THEN 'Beloit'
            WHEN ROUND(latitude, 4) = 42.6831 AND ROUND(longitude, 4) = -89.0038 THEN 'Janesville'
            WHEN ROUND(latitude, 4) = 43.0604 AND ROUND(longitude, 4) = -89.3995 THEN 'Madison'
            ELSE 'Unknown'
        END AS city,
        
        -- Window function to filter out duplicate rows within the source data itself
        ROW_NUMBER() OVER(
            PARTITION BY date, latitude, longitude 
            ORDER BY date DESC
        ) as row_num
    FROM weather_openmeteo.bronze.weather_raw
    -- Filter out records containing NULL values in key columns
    WHERE date IS NOT NULL 
      AND latitude IS NOT NULL 
      AND longitude IS NOT NULL
      AND temperature_2m IS NOT NULL
)

-- 2. Execute the idempotent Merge operation into the Silver table
MERGE INTO weather_clean AS target
USING (
    SELECT date, latitude, longitude, city, temperature 
    FROM source_cleaned 
    WHERE row_num = 1
) AS source
ON target.date = source.date 
   AND target.latitude = source.latitude 
   AND target.longitude = source.longitude

-- When the record already exists, update the mutable metrics and attributes
/*WHEN MATCHED THEN
  UPDATE SET
    target.city = source.city,
    target.temperature = source.temperature
*/
-- When the record is new, insert all columns into the target table
WHEN NOT MATCHED THEN
  INSERT (date, latitude, longitude, city, temperature)
  VALUES (source.date, source.latitude, source.longitude, source.city, source.temperature);
